In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CLIP_METADATA_PATH, PROCESSED_DIR
from src.video_utils import load_clip_metadata, validate_video_paths
from src.pose_extraction import initialize_pose_estimator
from src.sequence_builder import (
    ensure_output_directories,
    process_single_clip,
    build_all_sequences,
)

In [2]:
df = load_clip_metadata(CLIP_METADATA_PATH)
df_checked = validate_video_paths(df)

print("Number of clips:", len(df_checked))
print("All files exist:", df_checked["file_exists"].all())
df_checked.head()

Number of clips: 30
All files exist: True


,clip_id,video_path,label,absolute_path,file_exists
0,bad_jump_01,data/raw/bad/bad_jump_01.MOV,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,True
1,bad_jump_02,data/raw/bad/bad_jump_02.MOV,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,True
2,bad_jump_03,data/raw/bad/bad_jump_03.MOV,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,True
3,bad_jump_04,data/raw/bad/bad_jump_04.MOV,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,True
4,bad_jump_05,data/raw/bad/bad_jump_05.MOV,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,True


In [3]:
ensure_output_directories()

In [4]:
bad_example = df_checked[df_checked["label"] == "bad_jump"].iloc[0]
good_example = df_checked[df_checked["label"] == "good_jump"].iloc[0]

test_df = pd.DataFrame([bad_example, good_example]).reset_index(drop=True)
test_df[["clip_id", "label", "absolute_path"]]

,clip_id,label,absolute_path
0,bad_jump_01,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...
1,good_jump_01,good_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...


In [5]:
pose_estimator = initialize_pose_estimator()

I0000 00:00:1775845430.361185 38851853 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [6]:
test_records = []

for _, row in test_df.iterrows():
    record = process_single_clip(
        clip_id=row["clip_id"],
        video_path=row["absolute_path"],
        label=row["label"],
        pose_estimator=pose_estimator,
        target_length=30,
    )
    test_records.append(record)
    print(f"Processed {row['clip_id']}")

W0000 00:00:1775845430.752439 38852097 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775845430.785063 38852101 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775845437.994828 38852102 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Processed bad_jump_01
Processed good_jump_01


In [7]:
pose_estimator.close()

In [8]:
test_processed_df = pd.DataFrame(test_records)
test_processed_df

,clip_id,video_path,label,sequence_path,num_sampled_frames,missing_pose_frames,processed_shape
0,bad_jump_01,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
1,good_jump_01,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,good_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,1,"(30, 132)"


In [9]:
sequence_dir = PROCESSED_DIR / "landmark_sequences"
saved_files = list(sequence_dir.glob("*.npy"))

print("Number of saved .npy files currently:", len(saved_files))
saved_files[:5]

Number of saved .npy files currently: 30


[PosixPath('/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/landmark_sequences/good_jump_14.npy'),
 PosixPath('/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/landmark_sequences/good_jump_15.npy'),
 PosixPath('/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/landmark_sequences/good_jump_01.npy'),
 PosixPath('/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/landmark_sequences/good_jump_03.npy'),
 PosixPath('/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/landmark_sequences/good_jump_02.npy')]

In [10]:
sample_sequence_path = test_processed_df.iloc[0]["sequence_path"]
loaded_sequence = np.load(sample_sequence_path)

print("Loaded sequence shape:", loaded_sequence.shape)
print("Contains NaNs:", np.isnan(loaded_sequence).any())
print("Contains infs:", np.isinf(loaded_sequence).any())

Loaded sequence shape: (30, 132)
Contains NaNs: False
Contains infs: False


In [11]:
processed_df = build_all_sequences(target_length=30)
processed_df.head()

I0000 00:00:1775845441.093231 38851853 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1775845441.191986 38852835 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775845441.204481 38852835 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Processed: bad_jump_01
Processed: bad_jump_02
Processed: bad_jump_03
Processed: bad_jump_04
Processed: bad_jump_05
Processed: bad_jump_06
Processed: bad_jump_07
Processed: bad_jump_08
Processed: bad_jump_09
Processed: bad_jump_10
Processed: bad_jump_11
Processed: bad_jump_12
Processed: bad_jump_13
Processed: bad_jump_14
Processed: bad_jump_15
Processed: good_jump_01
Processed: good_jump_02
Processed: good_jump_03
Processed: good_jump_04
Processed: good_jump_05
Processed: good_jump_06
Processed: good_jump_07
Processed: good_jump_08
Processed: good_jump_09
Processed: good_jump_10
Processed: good_jump_11
Processed: good_jump_12
Processed: good_jump_13
Processed: good_jump_14
Processed: good_jump_15


,clip_id,video_path,label,sequence_path,num_sampled_frames,missing_pose_frames,processed_shape
0,bad_jump_01,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
1,bad_jump_02,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
2,bad_jump_03,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
3,bad_jump_04,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,1,"(30, 132)"
4,bad_jump_05,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"


In [12]:
print("Number of processed clips:", len(processed_df))
print("\nClass counts:")
print(processed_df["label"].value_counts())

Number of processed clips: 30

Class counts:
label
bad_jump     15
good_jump    15
Name: count, dtype: int64


In [13]:
print(processed_df["missing_pose_frames"].describe())

count    30.000000
mean      0.100000
std       0.305129
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000
Name: missing_pose_frames, dtype: float64


In [14]:
processed_df["processed_shape"].value_counts()

processed_shape
(30, 132)    30
Name: count, dtype: int64

In [15]:
processed_metadata_path = PROCESSED_DIR / "processed_sequence_metadata.csv"
print(processed_metadata_path)
print(processed_metadata_path.exists())

/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/processed_sequence_metadata.csv
True


In [16]:
loaded_processed_meta = pd.read_csv(PROCESSED_DIR / "processed_sequence_metadata.csv")
loaded_processed_meta.head()

,clip_id,video_path,label,sequence_path,num_sampled_frames,missing_pose_frames,processed_shape
0,bad_jump_01,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
1,bad_jump_02,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
2,bad_jump_03,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
3,bad_jump_04,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,1,"(30, 132)"
4,bad_jump_05,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
